In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/ronaldleonardo/Data_Transformer_Datasets/refs/heads/main/bbc_text_cls.csv"
df = pd.read_csv(url)

df.head()

,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


In [2]:
df.tail()

,text,labels
2219,BT program to beat dialler scams\n\nBT is intr...,tech
2220,Spam e-mails tempt net shoppers\n\nComputer us...,tech
2221,Be careful how you code\n\nA new European dire...,tech
2222,US cyber security chief resigns\n\nThe man mak...,tech
2223,Losing yourself in online gaming\n\nOnline rol...,tech


In [4]:
print(df.shape)
print(df.columns)
print(df["labels"].value_counts())

(2224, 2)
Index(['text', 'labels'], dtype='str')
labels
business         510
sport            510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


In [5]:
# Check missing values
print(df.isnull().sum())

# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

text      0
labels    0
dtype: int64
Duplicate rows: 98


# Step 1: Text cleaning

In [7]:
import re
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")

# Initialize
sw = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

keep_words = {
    "no", "nor", "not", "never",
    "couldn't", "didn't", "doesn't", "don't",
    "isn't", "wasn't", "weren't", "won't",
    "wouldn't", "shouldn't", "haven't", "hasn't"
}

sw -= keep_words

# Convert POS tags to WordNet tags
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def clean_text(text):
    # Handle missing or non-string values
    if not isinstance(text, str):
        return ""

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Tokenize while preserving contractions
    words = re.findall(r"\b[a-z]+(?:'[a-z]+)?\b", text)

    # 3. Remove stopwords, preserving selected negations
    words = [word for word in words if word not in sw]

    # 4. POS tagging for lemmatization
    tagged_words = pos_tag(words)

    # 5. Lemmatize without changing preserved negations
    cleaned_words = [
        word if word in keep_words
        else lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged_words
    ]

    # 6. Join tokens
    return " ".join(cleaned_words)

# Apply cleaning
df["clean_text"] = df["text"].apply(clean_text)

# Verify
df[["text", "clean_text", "labels"]].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/raviguna/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/raviguna/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/raviguna/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


,text,clean_text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,ad sale boost time warner profit quarterly pro...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,dollar gain greenspan speech dollar hit high l...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,yukos unit buyer face loan claim owner embattl...,business
3,High fuel prices hit BA's profits\n\nBritish A...,high fuel price hit ba's profit british airway...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,pernod takeover talk lift domecq share uk drin...,business


# Train/test split

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["clean_text"]
y = df["labels"]

# 1. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 2. Apply TF-IDF
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Naive Bayes

In [10]:
from sklearn.naive_bayes import MultinomialNB

# Initialize model
nb = MultinomialNB()

# Train model
nb.fit(X_train_tfidf, y_train)

# Predict test data
y_pred = nb.predict(X_test_tfidf)

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9842696629213483

Classification Report:
               precision    recall  f1-score   support

     business       0.98      0.98      0.98       102
entertainment       1.00      0.96      0.98        77
     politics       0.98      0.99      0.98        84
        sport       1.00      1.00      1.00       102
         tech       0.96      0.99      0.98        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445


Confusion Matrix:
[[100   0   1   0   1]
 [  1  74   1   0   1]
 [  0   0  83   0   1]
 [  0   0   0 102   0]
 [  1   0   0   0  79]]


In [12]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

gnb = GaussianNB()

# GaussianNB requires dense arrays
gnb.fit(X_train_tfidf.toarray(), y_train)

y_pred_gnb = gnb.predict(X_test_tfidf.toarray())

print("Gaussian NB Accuracy:", accuracy_score(y_test, y_pred_gnb))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_gnb))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_gnb))

Gaussian NB Accuracy: 0.9213483146067416

Classification Report:

               precision    recall  f1-score   support

     business       0.96      0.84      0.90       102
entertainment       0.87      0.97      0.92        77
     politics       0.92      0.93      0.92        84
        sport       0.97      0.95      0.96       102
         tech       0.88      0.93      0.90        80

     accuracy                           0.92       445
    macro avg       0.92      0.92      0.92       445
 weighted avg       0.92      0.92      0.92       445


Confusion Matrix:

[[86  4  4  1  7]
 [ 0 75  1  0  1]
 [ 0  3 78  2  1]
 [ 3  1  0 97  1]
 [ 1  3  2  0 74]]


In [14]:
import joblib

joblib.dump(nb, "news_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!
